# Parse Landscape Plants PDF

Use `ai_parse_document` to extract structured content from the native and adapted landscape plants PDF.

In [0]:
plant_df = spark.sql(
    """
    WITH plant_data AS (
        SELECT
            path,
            ai_parse_document(content, MAP('version', '2.0')) AS parsed
        FROM READ_FILES(
            '/Volumes/powers_land_catalog/natives/plant_chars/Native-and-adapted-Landscape-Plants.pdf'
            ,format => 'binaryFile')
    )
    SELECT
        path,
        parsed,
        parsed:document:pages,
        parsed:document:elements,
        parsed:error_status,
        parsed:metadata
    FROM plant_data;
    """
)


In [0]:
plant_df.write.mode(
    "overwrite"
    ).saveAsTable(
        "powers_land_catalog.natives.raw_parse_texas_agrilife_native_plants"
        )

In [0]:
%sql
SELECT * FROM powers_land_catalog.natives.raw_parse_texas_agrilife_native_plants

In [0]:
plant_ai_extract = spark.sql("""
    WITH elements AS (
        SELECT
            posexplode(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) AS (pos, el)
        FROM powers_land_catalog.natives.raw_parse_texas_agrilife_native_plants
    ),
    with_group AS (
        SELECT
            pos,
            el,
            el:type::STRING AS element_type,
            last_value(
            CASE WHEN el:type::STRING = 'section_header' THEN el:content::STRING END, true
            ) OVER (ORDER BY pos ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS plant_group
        FROM elements
    ),
    table_elements AS (
        SELECT plant_group, el AS table_el
        FROM with_group
        WHERE element_type = 'table'
    ),
    extracted AS (
        SELECT
            plant_group,
            ai_extract(
            table_el,
            '{
                "plants": {
                "type": "array",
                "description": "Each row in this HTML table is a plant",
                "items": {
                    "type": "object",
                    "properties": {
                    "common_name": {"type": "string"},
                    "botanical_name": {"type": "string", "description": "Scientific/latin name, often in italics"},
                    "texas_native": {"type": "string", "description": "B=Both native & adapted, E=Endemic/native, T=Adapted from Texas region"},
                    "height": {"type": "string"},
                    "spread": {"type": "string"},
                    "light": {"type": "string"},
                    "decid_evergreen": {"type": "string", "description": "E=Evergreen, SE=Semi-Evergreen, D=Deciduous"},
                    "seasonal_interest": {"type": "string"},
                    "color_feature": {"type": "string"},
                    "water": {"type": "string", "description": "VL=Very Low, L=Low, M=Medium, H=High"},
                    "wildlife": {"type": "string"},
                    "comments_maintenance": {"type": "string"}
                    }
                }
                }
            }',
            MAP(
                'version', '2.1',
                'instructions', 'Each row in the HTML table is one plant. The first column usually combines common name and botanical name. Extract every row as a separate plant.'
            )
            ) AS extracted
        FROM table_elements
    )
    SELECT * FROM extracted;
""")


In [0]:

plant_ai_extract.write.mode("overwrite").saveAsTable("powers_land_catalog.natives.raw_extract_texas_agrilife_native_plants")


In [0]:
%sql
SELECT * FROM powers_land_catalog.natives.raw_extract_texas_agrilife_native_plants